# Parte 1 – Scraping de decretos de emergencia

**Grupo 3** · Temporada: **1 de enero al 31 de mayo de 2024**

Reunimos los decretos supremos de la PCM que declaran o prorrogan estados de emergencia por lluvias en nuestra temporada. Estos datos sirven para responder la pregunta del trabajo: *¿el Estado declara emergencia donde más llueve?*

| Pasos | Contenido |
|---|---|
| 1 | Revisión de `robots.txt` |
| 2 – 5 | Búsqueda mes por mes con Selenium |
| 6 | Verificación: resultados anunciados frente a extraídos |
| 7 – 8 | Normas repetidas y solo normas de la PCM |
| 9 | Títulos completos con `requests` y `BeautifulSoup` |
| 10 – 11 | Clasificación (`es_lluvia`, `tipo`, `motivo`) y normas de lluvias |
| 12 | Departamentos que menciona cada decreto |
| 13 | Archivos en `datos/` |

### 1. Descarguen e impriman el archivo `https://www.gob.pe/robots.txt`. En una celda de texto, expliquen **qué prohíbe**, si la página de búsquedas (`/busquedas`) está permitida y **por qué igual conviene hacer pausas** entre página y página (fíjense en el `Crawl-delay` que les piden a otros bots).

In [47]:
# Importa la biblioteca requests, que permite realizar solicitudes a páginas web.
import requests

# Define una cabecera que identifica nuestra solicitud como proveniente de un navegador.
headers = {"User-Agent": "Mozilla/5.0"}

# Realiza una solicitud HTTP para obtener el archivo robots.txt de gob.pe.
robots = requests.get(

    # Indica la dirección web del archivo que queremos consultar.
    "https://www.gob.pe/robots.txt",

    # Envía la cabecera que definimos anteriormente.
    headers=headers,

    # Establece un tiempo máximo de espera de 30 segundos.
    timeout=30
)

# Imprime los primeros 600 caracteres del contenido recibido.
print(robots.text[:600])

# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



**a) ¿Qué prohíbe?**

Las reglas de `User-agent: *` se aplican a todos los robots, incluido nuestro código:

- `Disallow: /admin/`: no se debe entrar en la sección administrativa del sitio.
- `Disallow: /*?sheet=` y `Disallow: /*&sheet=`: no se deben visitar direcciones que usan el parámetro `sheet`.

Además, `User-agent: AhrefsBot` con `Disallow: /` prohíbe a ese robot rastrear cualquier página del sitio.

**b) ¿Está permitida la página de búsquedas?**

Sí. Ninguna regla menciona `/busquedas`, así que está permitida.

**c) ¿Por qué conviene hacer pausas entre páginas?**

El sitio pide pausas a otros bots: `Crawl-delay: 5` a GPTBot (5 segundos entre peticiones) y `Crawl-delay: 2` a OAI-SearchBot (2 segundos). Eso indica que gob.pe limita la frecuencia de consultas. Por eso hacemos pausas entre páginas, para no sobrecargar el servidor.

### 2. Con Selenium, abran la búsqueda de **un solo mes** de su temporada. Usen `WebDriverWait`, no solo `time.sleep()`.

In [48]:
# Instala la biblioteca Selenium.
#%pip install selenium

In [49]:
# Crea una variable llamada url_base para guardar la dirección del buscador.
url_base = (

    # Indica la dirección principal del buscador de gob.pe.
    "https://www.gob.pe/busquedas?"

    # Limita la búsqueda a contenidos clasificados como normas.
    "contenido[]=normas&"

    # Aplica el filtro de institución: PCM.
    "institucion[]=pcm&"

    # Establece los términos que queremos buscar.
    # %20 representa un espacio dentro de la dirección web.
    "term=estado%20de%20emergencia%20precipitaciones&"

    # Ordena los resultados comenzando por los más recientes.
    "sort_by=recent"
)


# Crea una nueva URL añadiendo a la búsqueda el periodo de enero de 2024.
url_enero = url_base + "&desde=01-01-2024&hasta=31-01-2024"


# Muestra la URL completa para comprobar que se construyó correctamente.
print(url_enero)

https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-01-2024&hasta=31-01-2024


In [50]:
# Importa webdriver, que permite controlar un navegador web desde Python.
from selenium import webdriver

# Importa By, que permite indicar cómo localizar elementos en una página web.
from selenium.webdriver.common.by import By

# Importa WebDriverWait, que permite esperar hasta que se cumpla una condición.
from selenium.webdriver.support.ui import WebDriverWait

# Importa las condiciones que podemos utilizar durante la espera.
# EC es un nombre abreviado de expected_conditions.
from selenium.webdriver.support import expected_conditions as EC


# Inicia una nueva sesión de Chrome controlada por Selenium.
# Guarda el navegador en la variable "navegador".
navegador = webdriver.Chrome()


# Ordena al navegador que abra la URL de enero que construimos anteriormente.
navegador.get(url_enero)


# Establece una espera de hasta 30 segundos para que se cumpla una condición.
WebDriverWait(navegador, 30).until(

    # Define la condición: debe existir un elemento que coincida con nuestra búsqueda.
    EC.presence_of_element_located(

        # Indica que buscaremos el elemento mediante un selector CSS.
        # "article" es la etiqueta HTML que contiene cada resultado de búsqueda.
        (By.CSS_SELECTOR, "article")
    )
)


# Muestra un mensaje cuando se ha encontrado al menos un elemento "article".
print("La página de búsqueda ha cargado y se encontró un resultado.")

La página de búsqueda ha cargado y se encontró un resultado.


In [51]:
# Establece una espera máxima de 30 segundos en el navegador.
WebDriverWait(navegador, 30).until(

    # Indica que la espera terminará cuando se encuentre el elemento solicitado.
    EC.presence_of_element_located(

        # Busca un elemento HTML llamado "article" mediante un selector CSS.
        (By.CSS_SELECTOR, "article")
    )
)

<selenium.webdriver.remote.webelement.WebElement (session="b58e199bf008d84340e1886e607f6e78", element="f.3BFA20EDA1F98F804CFF6B25EB268C2B.d.F971A8B331E0BBEC3B7BD123B09F0322.e.10")>

In [52]:
# Muestra la dirección web de la página que Chrome tiene abierta.
print("URL actual:", navegador.current_url)


# Busca todos los elementos HTML llamados "article".
# Guarda los elementos encontrados en una lista llamada "articulos".
articulos = navegador.find_elements(By.CSS_SELECTOR, "article")


# Cuenta cuántos elementos hay en la lista "articulos".
# Muestra esa cantidad en pantalla.
print("Número de artículos encontrados:", len(articulos))


# Comprueba si la lista "articulos" contiene al menos un elemento.
if articulos:

    # Imprime un salto de línea y un título para identificar el contenido.
    print("\nContenido del primer artículo:")

    # Selecciona el primer artículo, obtiene su texto y muestra sus primeros 500 caracteres.
    print(articulos[0].text[:500])

URL actual: https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-01-2024&hasta=31-01-2024
Número de artículos encontrados: 11

Contenido del primer artículo:
PCM - Presidencia del Consejo de Ministros
Normas y documentos legales
›
Decretos
Decreto Supremo N.° 010-2024-PCM
Publicado: 30 de enero de 2024
Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrant ...
Descargar archivo


In [53]:
# Comprueba si Python todavía puede comunicarse con Chrome.
print(navegador.current_url)

https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-01-2024&hasta=31-01-2024


In [54]:
# Busca el cuerpo de la página y guarda el elemento encontrado.
cuerpo = navegador.find_element(By.TAG_NAME, "body")

# Comprueba que Selenium encontró el elemento.
print("Se encontró el cuerpo de la página.")

Se encontró el cuerpo de la página.


In [55]:
# Muestra los primeros 1000 caracteres del texto visible de la página.
print(cuerpo.text[:1000])

Saltar a contenido principal
Inicio
El Estado
PCM
Resultado de búsquedas
Buscar en Gob.pe
Por institución:
Todas las instituciones
Por fecha de publicación:
01/01/24 - 31/01/24
Todo
Orientación, trámites y servicios
Noticias
Campañas
Documentos y publicaciones
Normativa
Información institucional
Filtros(2)
11 Resultados
Ordenar por:
Más recientes
PCM - Presidencia del Consejo de Ministros
Normas y documentos legales
›
Decretos
Decreto Supremo N.° 010-2024-PCM
Publicado: 30 de enero de 2024
Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrant ...
Descargar archivo
PCM - Presidencia del Consejo de Ministros
Normas y documentos legales
›
Decretos
Decreto Supremo N.° 009-2024-PCM
Publicado: 29 de enero de 2024
Prorrogar por el termino de sesenta (60) días calendario, a partir del 1 de febrero de 2024, el Estado de Emergencia declarado en los dis

### 3. Lean el número total de resultados que muestra la página ("N Resultados").

**Objetivo:** extraer el número total de resultados que anuncia el buscador de gob.pe para enero de 2024.

**Estrategia:** buscamos en el texto visible de la página la expresión «N Resultados», extraemos el número con una expresión regular y lo guardamos en `total_resultados`. Ese valor sirve después para comprobar que extrajimos todos los registros.

In [56]:
# Importa la biblioteca para buscar patrones dentro de un texto.
import re

# Busca un número seguido de la palabra "Resultados".
coincidencia = re.search(r"(\d+)\s+Resultados", cuerpo.text)

# Comprueba si se encontró el patrón esperado.
if coincidencia:

    # Extrae el número encontrado y lo convierte en un entero.
    total_resultados = int(coincidencia.group(1))

    # Muestra el total anunciado por la página.
    print("Total anunciado:", total_resultados)

    # Muestra cuántos artículos encontró Selenium.
    print("Artículos encontrados:", len(articulos))

    # Comprueba si ambas cantidades coinciden.
    print("¿Coinciden?", total_resultados == len(articulos))

else:

    # Informa si no se encontró el texto esperado.
    print("No se encontró el total de resultados.")

Total anunciado: 11
Artículos encontrados: 11
¿Coinciden? True


### 4. De cada resultado, extraigan:

- número de la norma: es el texto del enlace principal (por ejemplo: `Decreto Supremo N.° 044-2023-PCM`);
- fecha de publicación: el texto que empieza con `Publicado:`;
- título: el párrafo debajo de la fecha;
- enlace a la norma.

> Pista: cada resultado es un `<article>`. Primero encuentren todos los `article` y luego busquen cada dato dentro de cada uno.

In [57]:
# Guarda el primer artículo de la búsqueda de enero.
primer_articulo = articulos[0]

# Busca todos los enlaces que están dentro del primer artículo.
enlaces = primer_articulo.find_elements(By.CSS_SELECTOR, "a[href]")

# Recorre los enlaces encontrados, uno por uno.
for enlace in enlaces:

    # Muestra el texto visible del enlace.
    print("Texto:", enlace.text)

    # Muestra la dirección web a la que conduce.
    print("URL:", enlace.get_attribute("href"))

    # Imprime una línea para separar los enlaces.
    print("---")

Texto: Decreto Supremo N.° 010-2024-PCM
URL: https://www.gob.pe/institucion/pcm/normas-legales/5398152-010-2024-pcm
---
Texto: Descargar archivo
URL: https://cdn.www.gob.pe/uploads/document/file/6103394/5398152-ds-n-0010-2024-pcm.pdf
---


In [58]:
# Obtiene el texto visible del primer artículo.
texto_articulo = primer_articulo.text

# Divide el texto del artículo en líneas.
lineas = texto_articulo.splitlines()

# Recorre las líneas para identificar dónde están los datos.
for numero, linea in enumerate(lineas):

    # Muestra el número de línea y su contenido.
    print(numero, ":", linea)

0 : PCM - Presidencia del Consejo de Ministros
1 : Normas y documentos legales
2 : ›
3 : Decretos
4 : Decreto Supremo N.° 010-2024-PCM
5 : Publicado: 30 de enero de 2024
6 : Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrant ...
7 : Descargar archivo


In [59]:
# Busca el enlace principal del primer artículo.
enlace_principal = primer_articulo.find_element(
    By.CSS_SELECTOR, "a[href*='/institucion/pcm/normas-legales/']"
)

# Obtiene el número de la norma a partir del texto del enlace.
numero_norma = enlace_principal.text

# Obtiene la dirección web de la norma.
enlace_norma = enlace_principal.get_attribute("href")

# Busca la línea que comienza con "Publicado:".
linea_fecha = next(
    linea for linea in lineas if linea.startswith("Publicado:")
)

# Elimina la palabra "Publicado:" para conservar únicamente la fecha.
fecha_publicacion = linea_fecha.replace("Publicado:", "").strip()

# Obtiene el título que aparece inmediatamente después de la fecha.
titulo = lineas[lineas.index(linea_fecha) + 1]

# Muestra los cuatro datos extraídos.
print("Número:", numero_norma)
print("Fecha:", fecha_publicacion)
print("Título:", titulo)
print("Enlace:", enlace_norma)

Número: Decreto Supremo N.° 010-2024-PCM
Fecha: 30 de enero de 2024
Título: Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrant ...
Enlace: https://www.gob.pe/institucion/pcm/normas-legales/5398152-010-2024-pcm


In [60]:
# Crea una lista vacía donde guardaremos los decretos.
decretos_enero = []

# Recorre los artículos encontrados en la búsqueda de enero.
for articulo in articulos:

    # Busca el enlace principal de la norma dentro del artículo.
    enlace_principal = articulo.find_element(
        By.CSS_SELECTOR, "a[href*='/normas-legales/']"
    )

    # Obtiene el número de la norma a partir del texto del enlace.
    numero_norma = enlace_principal.text

    # Obtiene la dirección web de la norma.
    enlace_norma = enlace_principal.get_attribute("href")

    # Obtiene el texto visible del artículo y lo divide en líneas.
    lineas = articulo.text.splitlines()

    # Busca la línea que comienza con la palabra "Publicado:".
    linea_fecha = next(
        linea for linea in lineas if linea.startswith("Publicado:")
    )

    # Elimina la palabra "Publicado:" para conservar la fecha.
    fecha_publicacion = linea_fecha.replace("Publicado:", "").strip()

    # Obtiene el título que aparece después de la fecha.
    titulo = lineas[lineas.index(linea_fecha) + 1]

    # Organiza los cuatro datos del decreto en un diccionario.
    decreto = {
        "numero": numero_norma,
        "fecha": fecha_publicacion,
        "titulo": titulo,
        "enlace": enlace_norma
    }

    # Añade el diccionario a nuestra lista de decretos.
    decretos_enero.append(decreto)

# Muestra cuántos decretos hemos guardado.
print("Decretos extraídos:", len(decretos_enero))

Decretos extraídos: 11


In [61]:
# Importa pandas para organizar los datos en una tabla.
import pandas as pd

# Convierte la lista de decretos en una tabla.
tabla_enero = pd.DataFrame(decretos_enero)

# Muestra las primeras cinco filas de la tabla.
display(tabla_enero)

# Comprueba si extrajimos la cantidad anunciada por el buscador.
print("¿Coinciden las cantidades?", len(tabla_enero) == total_resultados)

# Comprueba si hay datos faltantes en alguna columna.
print("Datos faltantes por columna:")
print(tabla_enero.isna().sum())

,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 010-2024-PCM,30 de enero de 2024,"Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrant ...",https://www.gob.pe/institucion/pcm/normas-legales/5398152-010-2024-pcm
1,Decreto Supremo N.° 009-2024-PCM,29 de enero de 2024,"Prorrogar por el termino de sesenta (60) días calendario, a partir del 1 de febrero de 2024, el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la p ...",https://www.gob.pe/institucion/pcm/normas-legales/5398113-009-2024-pcm
2,Decreto Supremo N.° 008-2024-PCM,29 de enero de 2024,"Declarar por el termino de treinta (30) días calendario, a partir del 5 de febrero de 2024 el Estado de Emergencia en el Corredor Vial Sur Apurímac-Cusco-Arequipa, incluyendo los quinientos (500) met ...",https://www.gob.pe/institucion/pcm/normas-legales/5398076-008-2024-pcm
3,Decreto Supremo N.° 007-2024-PCM,29 de enero de 2024,"Decreto Supremo que prorroga el Estado de Emergencia en el distrito de Echarate de la provincia de La Convención del departamento de Cusco, por impacto de daños a consecuencia de intensas pre ...",https://www.gob.pe/institucion/pcm/normas-legales/5225977-007-2024-pcm
4,Decreto Supremo N.° 006-2024-PCM,29 de enero de 2024,"Decreto Supremo que prorroga el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Arequipa, Ayacucho, Cajamarca, Ica, La Libertad, Lambayeque, L ...",https://www.gob.pe/institucion/pcm/normas-legales/5225965-006-2024-pcm
5,Decreto Supremo N.° 005-2024-PCM,29 de enero de 2024,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Arequipa, Ayacucho, Cusco, Huánuco, Huancavelica, Ica, Moquegua y Pasco, por impa ...",https://www.gob.pe/institucion/pcm/normas-legales/5225954-005-2024-pcm
6,Decreto Supremo N.° 004-2024-PCM,26 de enero de 2024,"Decreto Supremo que declara el Estado de Emergencia en el distrito de Chalhuanca de la provincia de Aymaraes del departamento de Apurímac, por impacto de daños a consecuencia de intensas prec ...",https://www.gob.pe/institucion/pcm/normas-legales/5225947-004-2024-pcm
7,Resolución N.° 0214-2024-TCE-S6,18 de enero de 2024,"N° 10304103618); por su responsabilidad al haber contratado con el Estado estando impedida para ello, en el marco de la Orden de Servicio: N° 160-2019-Área de Abastecimiento de fecha 12 de febrero de 2019, emitida po ...",https://www.gob.pe/institucion/oece/normas-legales/5187307-0214-2024-tce-s6
8,Decreto Supremo N.° 003-2024-PCM,10 de enero de 2024,"Decreto Supremo que declara el Estado de Emergencia en la provincia de Zarumilla del departamento de Tumbes; en las provincias de Sullana, Ayabaca y Huancabamba del departamento de Piura; en la provi ...",https://www.gob.pe/institucion/pcm/normas-legales/5225938-003-2024-pcm
9,Decreto Supremo N.° 002-2024-PCM,9 de enero de 2024,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Pataz del departamento de La Libertad,https://www.gob.pe/institucion/pcm/normas-legales/5225914-002-2024-pcm


¿Coinciden las cantidades? True
Datos faltantes por columna:
numero    0
fecha     0
titulo    0
enlace    0
dtype: int64


### 5. Repitan los pasos 2 a 4 para **todos los meses** de su temporada. Hagan una pausa de al menos 2 segundos entre cada página.

Enero ya se trabajó en los pasos 2 a 4. Aquí siguen febrero, marzo, abril y mayo.

#### Febrero

**Abrir la búsqueda de febrero**

In [62]:
# Construye la URL de febrero utilizando la URL base del profesor.
url_febrero = url_base + "&desde=01-02-2024&hasta=29-02-2024"

# Abre en Chrome la búsqueda correspondiente a febrero.
navegador.get(url_febrero)

# Espera hasta 30 segundos a que aparezca el cuerpo de la página.
WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.TAG_NAME, "body"))
)

# Muestra la dirección para comprobar que estamos en febrero.
print("URL actual:", navegador.current_url)

URL actual: https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-02-2024&hasta=29-02-2024


**Total anunciado y artículos en la página**

In [63]:
# Obtiene el texto visible de la página.
texto_febrero = navegador.find_element(By.TAG_NAME, "body").text

# Busca el número que aparece antes de la palabra "Resultados".
coincidencia_febrero = re.search(
    r"(\d+)\s+Resultados",
    texto_febrero
)

# Comprueba si encontramos el total anunciado.
if coincidencia_febrero:

    # Convierte el número encontrado en un entero.
    total_febrero = int(coincidencia_febrero.group(1))

    # Muestra el total anunciado por el buscador.
    print("Total anunciado:", total_febrero)

else:

    # Informa si no encontramos el total.
    print("No se encontró el total anunciado.")

# Busca los artículos presentes en la página.
articulos_febrero = navegador.find_elements(By.CSS_SELECTOR, "article")

# Muestra cuántos artículos encontró Selenium.
print("Artículos en la página:", len(articulos_febrero))

Total anunciado: 11
Artículos en la página: 11


**¿Hay paginación?** Buscamos enlaces o botones como «siguiente» o «página» para saber si todos los resultados están en una sola página.

In [64]:
# Busca todos los enlaces de la página que tienen una dirección web.
enlaces_pagina = navegador.find_elements(By.CSS_SELECTOR, "a[href]")

# Recorre los enlaces encontrados.
for enlace in enlaces_pagina:

    # Obtiene el texto visible del enlace.
    texto_enlace = enlace.text.strip()

    # Obtiene la dirección web del enlace.
    direccion = enlace.get_attribute("href")

    # Comprueba si el enlace parece estar relacionado con la paginación.
    if (
        "siguiente" in texto_enlace.lower()
        or "página" in texto_enlace.lower()
        or "page=" in direccion.lower()
    ):

        # Muestra el texto del enlace.
        print("Texto:", texto_enlace)

        # Muestra su dirección web.
        print("URL:", direccion)

        # Separa visualmente los enlaces encontrados.
        print("---")

In [65]:
# Busca elementos que podrían utilizarse para navegar entre páginas.
controles = navegador.find_elements(
    By.CSS_SELECTOR,
    "a, button"
)

# Recorre los controles encontrados.
for control in controles:

    # Obtiene el texto visible del control.
    texto = control.text.strip()

    # Obtiene su descripción, si existe.
    descripcion = control.get_attribute("aria-label") or ""

    # Comprueba si el texto o la descripción mencionan la navegación.
    if any(
        palabra in (texto + " " + descripcion).lower()
        for palabra in ["siguiente", "anterior", "página", "paginación"]
    ):

        # Muestra el texto del control.
        print("Texto:", texto)

        # Muestra su descripción.
        print("Descripción:", descripcion)

        # Muestra el tipo de elemento HTML.
        print("Elemento:", control.tag_name)

        # Separa los resultados.
        print("---")

**Extracción de los decretos de febrero**

In [66]:
# Crea una lista vacía para guardar los decretos de febrero.
decretos_febrero = []

# Recorre los artículos encontrados en febrero.
for articulo in articulos_febrero:

    # Busca el enlace principal de la norma.
    enlace_principal = articulo.find_element(
        By.CSS_SELECTOR, "a[href*='/normas-legales/']"
    )

    # Obtiene el número de la norma.
    numero_norma = enlace_principal.text

    # Obtiene la dirección web de la norma.
    enlace_norma = enlace_principal.get_attribute("href")

    # Obtiene el texto del artículo y lo separa en líneas.
    lineas = articulo.text.splitlines()

    # Busca la línea que contiene la fecha de publicación.
    linea_fecha = next(
        linea for linea in lineas if linea.startswith("Publicado:")
    )

    # Elimina la palabra "Publicado:" para conservar la fecha.
    fecha_publicacion = linea_fecha.replace("Publicado:", "").strip()

    # Obtiene el título situado inmediatamente después de la fecha.
    titulo = lineas[lineas.index(linea_fecha) + 1]

    # Organiza los cuatro datos en un diccionario.
    decreto = {
        "numero": numero_norma,
        "fecha": fecha_publicacion,
        "titulo": titulo,
        "enlace": enlace_norma
    }

    # Guarda el decreto en la lista de febrero.
    decretos_febrero.append(decreto)

# Muestra la cantidad de decretos extraídos.
print("Decretos extraídos en febrero:", len(decretos_febrero))

Decretos extraídos en febrero: 11


**Verificación:** cantidad extraída frente al total anunciado y datos faltantes.

In [67]:
# Convierte los decretos de febrero en una tabla.
tabla_febrero = pd.DataFrame(decretos_febrero)

# Muestra todos los registros de febrero.
display(tabla_febrero)

# Compara los registros extraídos con el total anunciado.
print("¿Coinciden las cantidades?", len(tabla_febrero) == total_febrero)

# Comprueba si alguna columna contiene valores faltantes.
print("Datos faltantes por columna:")
print(tabla_febrero.isna().sum())

,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 021-2024-PCM,28 de febrero de 2024,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Arequipa, Ica, Madre de Dios, Moquegua y Ucayali, por impacto de daños a consecue ...",https://www.gob.pe/institucion/pcm/normas-legales/5433283-021-2024-pcm
1,Decreto Supremo N.° 020-2024-PCM,26 de febrero de 2024,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, H ...",https://www.gob.pe/institucion/pcm/normas-legales/5533513-020-2024-pcm
2,Resolución N.° 0634-2024-TCE-S5,23 de febrero de 2024,"Recurso de apelación interpuesto por la empresa SIMED PERU S.A.C., en la ADJUDICACIÓN SIMPLIFICADA Nº 017-2023-HEAV-CS-1 - PRIMERA CONVOCATORIA, para la contratación de suministro de bienes: ""Suministro para el siste ...",https://www.gob.pe/institucion/oece/normas-legales/5260816-0634-2024-tce-s5
3,Resolución N.° 0533-2024-TCE-S4,14 de febrero de 2024,"Recurso de apelación interpuesto por la empresa DIAGNOSTICA PERUANA S.A.C., en el marco de la Licitación Pública Nº 20-2022-ESSALUD-RPR - Primera Convocatoria, convocada por el Seguro Social de Salud, para la ""Adquis ...",https://www.gob.pe/institucion/oece/normas-legales/5197999-0533-2024-tce-s4
4,Decreto Supremo N.° 019-2024-PCM,13 de febrero de 2024,Decreto Supremo que declara el Estado de Emergencia en la provincia de Trujillo del departamento de La Libertad,https://www.gob.pe/institucion/pcm/normas-legales/5533472-019-2024-pcm
5,Decreto Supremo N.° 018-2024-PCM,13 de febrero de 2024,Decreto Supremo que declara el Estado de Emergencia en la provincia de Pataz del departamento de La Libertad,https://www.gob.pe/institucion/pcm/normas-legales/5533404-018-2024-pcm
6,Resolución N.° 0506-2024-TCE-S5,9 de febrero de 2024,"Recurso de apelación interpuesto por la empresa MEDICALAB SOCIEDAD ANONIMA CERRADA, en el marco de la Adjudicación Simplificada N° 1-2023-CS/HSMSI-1, convocada por el HOSPITAL DE APOYO SANTA MARIA DEL SOCORRO, para l ...",https://www.gob.pe/institucion/oece/normas-legales/5193883-0506-2024-tce-s5
7,Decreto Supremo N.° 016-2024-PCM,9 de febrero de 2024,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cusco, Huancavelica, Huánuco, Jun ...",https://www.gob.pe/institucion/pcm/normas-legales/5533271-016-2024-pcm
8,Resolución N.° 0437-2024-TCE-S3,5 de febrero de 2024,"Braun Medical Perú S.A. contra el otorgamiento de la buena pro de la Licitación pública N.º 09-2023-HMA-1 – Primera convocatoria, convocado por el Hospital de Emergencias Villa El Salvador",https://www.gob.pe/institucion/oece/normas-legales/5192380-0437-2024-tce-s3
9,Decreto Supremo N.° 013-2024-PCM,5 de febrero de 2024,Decreto Supremo que prorroga el Estado de Emergencia en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto,https://www.gob.pe/institucion/pcm/normas-legales/5532468-013-2024-pcm


¿Coinciden las cantidades? True
Datos faltantes por columna:
numero    0
fecha     0
titulo    0
enlace    0
dtype: int64


#### Marzo

**Abrir la búsqueda de marzo**

In [68]:
# Construye la URL de marzo de 2024.
url_marzo = url_base + "&desde=01-03-2024&hasta=31-03-2024"

# Abre la búsqueda de marzo en Chrome.
navegador.get(url_marzo)

# Espera hasta 30 segundos a que aparezca el cuerpo de la página.
WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.TAG_NAME, "body"))
)

# Muestra la URL para comprobar que corresponde a marzo.
print("URL actual:", navegador.current_url)

URL actual: https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-03-2024&hasta=31-03-2024


**Total anunciado y artículos en la página**

In [69]:
# Obtiene el texto visible de la página.
texto_marzo = navegador.find_element(By.TAG_NAME, "body").text

# Busca el número que aparece antes de la palabra "Resultados".
coincidencia_marzo = re.search(r"(\d+)\s+Resultados", texto_marzo)

# Comprueba que se haya encontrado el total.
if coincidencia_marzo:

    # Convierte el total anunciado en un número entero.
    total_marzo = int(coincidencia_marzo.group(1))

    # Muestra el total anunciado por el buscador.
    print("Total anunciado:", total_marzo)

else:

    # Informa si no se encontró el total.
    print("No se encontró el total anunciado.")

# Busca los artículos cargados en la página.
articulos_marzo = navegador.find_elements(By.CSS_SELECTOR, "article")

# Muestra cuántos artículos encontró Selenium.
print("Artículos encontrados:", len(articulos_marzo))

Total anunciado: 11
Artículos encontrados: 11


**Extracción y verificación de los decretos de marzo**

In [70]:
# Crea una lista vacía para guardar los decretos de marzo.
decretos_marzo = []

# Recorre los artículos encontrados en la búsqueda de marzo.
for articulo in articulos_marzo:

    # Busca el enlace principal de la norma.
    enlace_principal = articulo.find_element(
        By.CSS_SELECTOR, "a[href*='/normas-legales/']"
    )

    # Obtiene el número de la norma.
    numero_norma = enlace_principal.text

    # Obtiene la dirección web de la norma.
    enlace_norma = enlace_principal.get_attribute("href")

    # Obtiene el texto del artículo y lo divide en líneas.
    lineas = articulo.text.splitlines()

    # Busca la línea que contiene la fecha de publicación.
    linea_fecha = next(
        linea for linea in lineas if linea.startswith("Publicado:")
    )

    # Elimina la palabra "Publicado:" para conservar la fecha.
    fecha_publicacion = linea_fecha.replace("Publicado:", "").strip()

    # Obtiene el título que aparece después de la fecha.
    titulo = lineas[lineas.index(linea_fecha) + 1]

    # Organiza los cuatro datos en un diccionario.
    decreto = {
        "numero": numero_norma,
        "fecha": fecha_publicacion,
        "titulo": titulo,
        "enlace": enlace_norma
    }

    # Añade el decreto a la lista de marzo.
    decretos_marzo.append(decreto)

# Convierte la lista de decretos en una tabla.
tabla_marzo = pd.DataFrame(decretos_marzo)

# Muestra los registros extraídos.
display(tabla_marzo)

# Comprueba si la cantidad extraída coincide con el total anunciado.
print("¿Coinciden las cantidades?", len(tabla_marzo) == total_marzo)

# Comprueba si existen valores faltantes.
print("Datos faltantes por columna:")
print(tabla_marzo.isna().sum())

,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 034-2024-PCM,27 de marzo de 2024,"Decreto Supremo que prorroga el Estado de Emergencia en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huepetuhe de l ...",https://www.gob.pe/institucion/pcm/normas-legales/5468918-034-2024-pcm
1,Decreto Supremo N.° 033-2024-PCM,26 de marzo de 2024,"Decreto Supremo que prorroga el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Arequipa, Ayacucho, Cusco, Huánuco, Huancavelica, Ica, Moquegua y Pasco, por imp ...",https://www.gob.pe/institucion/pcm/normas-legales/5468915-033-2024-pcm
2,Decreto Supremo N.° 032-2024-PCM,26 de marzo de 2024,"Decreto Supremo que prorroga el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Áncash, Apurímac, Cusco, Huancavelica, Lima, Loreto y San Martín, por impacto de ...",https://www.gob.pe/institucion/pcm/normas-legales/5468900-032-2024-pcm
3,Decreto Supremo N.° 031-2024-PCM,26 de marzo de 2024,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, H ...",https://www.gob.pe/institucion/pcm/normas-legales/5465813-031-2024-pcm
4,Resolución N.° 0991-2024-TCE-S6,21 de marzo de 2024,"Recurso de apelación interpuesto por el postor CORPORACIÓN ALIMENTARIA ALEXCAR S.A.C., en el marco de la Subasta Inversa Electrónica Nº 8-2023-HEJCU-1 (Primera Convocatoria) - Ítem N°s 4, 7, 8 y 9, convocada por el H ...",https://www.gob.pe/institucion/oece/normas-legales/5421041-0991-2024-tce-s6
5,Resolución N.° 0972-2024-TCE-S2,20 de marzo de 2024,"Recurso de apelación interpuesto por la empresa SERVICIOS SANTA GABRIELA S.A.C., en el marco de la Adjudicación Simplificada D.U. 032-2023 N° 065-2023-ENSA - Primera Convocatoria, para la “Adquisición, equipamiento, ...",https://www.gob.pe/institucion/oece/normas-legales/5420398-0972-2024-tce-s2
6,Decreto Supremo N.° 030-2024-PCM,19 de marzo de 2024,"Decreto Supremo que prorroga el Estado de Emergencia en el distrito de Chalhuanca de la provincia de Aymaraes del departamento de Apurímac, por impacto de daños a consecuencia de intensas pre ...",https://www.gob.pe/institucion/pcm/normas-legales/5434267-030-2024-pcm
7,Decreto Supremo N.° 027-2024-PCM,14 de marzo de 2024,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ju ...",https://www.gob.pe/institucion/pcm/normas-legales/5401492-027-2024-pcm
8,Decreto Supremo N.° 026-2024-PCM,7 de marzo de 2024,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Apurímac, Arequipa, Ayacucho, Cusco, Huancavelica, Huánuco, Junín, Lima ...",https://www.gob.pe/institucion/pcm/normas-legales/5534357-026-2024-pcm
9,Decreto Supremo N.° 025-2024-PCM,4 de marzo de 2024,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Zarumilla del departamento de Tumbes,https://www.gob.pe/institucion/pcm/normas-legales/5534315-025-2024-pcm


¿Coinciden las cantidades? True
Datos faltantes por columna:
numero    0
fecha     0
titulo    0
enlace    0
dtype: int64


#### Abril

**Abrir la búsqueda de abril**

In [71]:
# Importa la biblioteca que permite introducir pausas.
import time

# Construye la URL de abril de 2024.
url_abril = url_base + "&desde=01-04-2024&hasta=30-04-2024"

# Espera dos segundos antes de realizar otra consulta.
time.sleep(2)

# Abre la búsqueda de abril.
navegador.get(url_abril)

# Espera hasta que aparezcan los artículos de los resultados.
WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "article"))
)

# Muestra la dirección de la página abierta.
print("URL actual:", navegador.current_url)

URL actual: https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-04-2024&hasta=30-04-2024


**Total anunciado y artículos en la página**

In [72]:
# Obtiene el texto visible de la página.
texto_abril = navegador.find_element(By.TAG_NAME, "body").text

# Busca el número que aparece antes de la palabra "Resultados".
coincidencia_abril = re.search(r"(\d+)\s+Resultados", texto_abril)

# Obtiene el total anunciado por el buscador.
total_abril = int(coincidencia_abril.group(1))

# Encuentra los artículos cargados en la página.
articulos_abril = navegador.find_elements(By.CSS_SELECTOR, "article")

# Muestra el total anunciado.
print("Total anunciado:", total_abril)

# Muestra la cantidad de artículos encontrados.
print("Artículos encontrados:", len(articulos_abril))

Total anunciado: 15
Artículos encontrados: 15


**Extracción y verificación de los decretos de abril**

In [73]:
# Crea una lista vacía para guardar los decretos de abril.
decretos_abril = []

# Recorre los artículos encontrados en abril.
for articulo in articulos_abril:

    # Busca el enlace principal de la norma.
    enlace_principal = articulo.find_element(
        By.CSS_SELECTOR, "a[href*='/normas-legales/']"
    )

    # Obtiene el número de la norma.
    numero_norma = enlace_principal.text

    # Obtiene la dirección web de la norma.
    enlace_norma = enlace_principal.get_attribute("href")

    # Divide el texto del artículo en líneas.
    lineas = articulo.text.splitlines()

    # Identifica la línea que contiene la fecha de publicación.
    linea_fecha = next(
        linea for linea in lineas if linea.startswith("Publicado:")
    )

    # Extrae la fecha de publicación.
    fecha_publicacion = linea_fecha.replace("Publicado:", "").strip()

    # Obtiene el título que aparece después de la fecha.
    titulo = lineas[lineas.index(linea_fecha) + 1]

    # Organiza los datos del decreto en un diccionario.
    decreto = {
        "numero": numero_norma,
        "fecha": fecha_publicacion,
        "titulo": titulo,
        "enlace": enlace_norma
    }

    # Añade el decreto a la lista de abril.
    decretos_abril.append(decreto)

# Convierte los decretos de abril en una tabla.
tabla_abril = pd.DataFrame(decretos_abril)

# Muestra los registros extraídos.
display(tabla_abril)

# Comprueba si la cantidad extraída coincide con el total anunciado.
print("¿Coinciden las cantidades?", len(tabla_abril) == total_abril)

# Comprueba si existen valores faltantes.
print("Datos faltantes por columna:")
print(tabla_abril.isna().sum())

,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 049-2024-PCM,30 de abril de 2024,Decreto Supremo que declara el Estado de Emergencia en el distrito de Ayna de la provincia de La Mar y en el distrito de Santa Ana de Huaycahuacho de la provincia de Lucanas del departamento de Ayacu ...,https://www.gob.pe/institucion/pcm/normas-legales/5595005-049-2024-pcm
1,Decreto Supremo N.° 048-2024-PCM,30 de abril de 2024,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Zarumilla del departamento de Tumbes,https://www.gob.pe/institucion/pcm/normas-legales/5594998-048-2024-pcm
2,Decreto Supremo N.° 047-2024-PCM,30 de abril de 2024,Decreto Supremo que prorroga el Estado de Emergencia declarado en el Corredor Vial Sur Apurímac-Cusco-Arequipa,https://www.gob.pe/institucion/pcm/normas-legales/5594995-047-2024-pcm
3,Decreto Supremo N.° 046-2024-PCM,30 de abril de 2024,"Decreto Supremo que prorroga el Estado de Emergencia en el distrito de Mollepata de la provincia de Anta del departamento de Cusco, por impacto de daños a consecuencia de intensas precipitaci ...",https://www.gob.pe/institucion/pcm/normas-legales/5594298-046-2024-pcm
4,Decreto Supremo N.° 045-2024-PCM,25 de abril de 2024,"Decreto Supremo que prorroga el Estado de Emergencia en el distrito de Yura de la provincia de Arequipa del departamento de Arequipa, por impacto de daños a consecuencia de intensas precipita ...",https://www.gob.pe/institucion/pcm/normas-legales/5561360-045-2024-pcm
5,Decreto Supremo N.° 044-2024-PCM,25 de abril de 2024,"Decreto Supremo que prorroga el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Huancavelica, Huánuco, La Liber ...",https://www.gob.pe/institucion/pcm/normas-legales/5561341-044-2024-pcm
6,Decreto Supremo N.° 043-2024-PCM,15 de abril de 2024,"Declaratoria de Estado de Emergencia Declarar por el término de veinte (20) días calendario, el Estado de Emergencia en la provincia de Arequipa del departamento de Arequipa.",https://www.gob.pe/institucion/pcm/normas-legales/5698272-043-2024-pcm
7,Decreto Supremo N.° 042-2024-PCM,12 de abril de 2024,Decreto Supremo que prorroga el Estado de Emergencia en la provincia de Trujillo del departamento de La Libertad,https://www.gob.pe/institucion/pcm/normas-legales/5489312-042-2024-pcm
8,Decreto Supremo N.° 041-2024-PCM,12 de abril de 2024,Decreto Supremo que prorroga el Estado de Emergencia en la provincia de Pataz del departamento de La Libertad,https://www.gob.pe/institucion/pcm/normas-legales/5541614-041-2024-pcm
9,Decreto Supremo N.° 040-2024-PCM,11 de abril de 2024,Decreto Supremo que prorroga el Estado de Emergencia en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto,https://www.gob.pe/institucion/pcm/normas-legales/5489310-040-2024-pcm


¿Coinciden las cantidades? True
Datos faltantes por columna:
numero    0
fecha     0
titulo    0
enlace    0
dtype: int64


#### Mayo

**Abrir la búsqueda de mayo**

In [74]:
# Construye la URL de mayo de 2024.
url_mayo = url_base + "&desde=01-05-2024&hasta=31-05-2024"

# Espera dos segundos antes de realizar otra consulta.
time.sleep(2)

# Abre la búsqueda de mayo.
navegador.get(url_mayo)

# Espera hasta que aparezcan los artículos de los resultados.
WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "article"))
)

# Muestra la dirección de la página abierta.
print("URL actual:", navegador.current_url)

URL actual: https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-05-2024&hasta=31-05-2024


**Total anunciado y artículos en la página**

In [75]:
# Obtiene el texto visible de la página.
texto_mayo = navegador.find_element(By.TAG_NAME, "body").text

# Busca el número que aparece antes de la palabra "Resultados".
coincidencia_mayo = re.search(r"(\d+)\s+Resultados", texto_mayo)

# Obtiene el total anunciado por el buscador.
total_mayo = int(coincidencia_mayo.group(1))

# Encuentra los artículos cargados en la página.
articulos_mayo = navegador.find_elements(By.CSS_SELECTOR, "article")

# Muestra el total anunciado.
print("Total anunciado:", total_mayo)

# Muestra la cantidad de artículos encontrados.
print("Artículos encontrados:", len(articulos_mayo))

Total anunciado: 3
Artículos encontrados: 3


**Extracción y verificación de los decretos de mayo**

In [76]:
# Crea una lista vacía para guardar los decretos de mayo.
decretos_mayo = []

# Recorre los artículos encontrados en mayo.
for articulo in articulos_mayo:

    # Busca el enlace principal de la norma.
    enlace_principal = articulo.find_element(
        By.CSS_SELECTOR, "a[href*='/normas-legales/']"
    )

    # Obtiene el número de la norma.
    numero_norma = enlace_principal.text

    # Obtiene la dirección web de la norma.
    enlace_norma = enlace_principal.get_attribute("href")

    # Divide el texto del artículo en líneas.
    lineas = articulo.text.splitlines()

    # Identifica la línea que contiene la fecha de publicación.
    linea_fecha = next(
        linea for linea in lineas if linea.startswith("Publicado:")
    )

    # Extrae la fecha de publicación.
    fecha_publicacion = linea_fecha.replace("Publicado:", "").strip()

    # Obtiene el título que aparece después de la fecha.
    titulo = lineas[lineas.index(linea_fecha) + 1]

    # Organiza los datos del decreto en un diccionario.
    decreto = {
        "numero": numero_norma,
        "fecha": fecha_publicacion,
        "titulo": titulo,
        "enlace": enlace_norma
    }

    # Añade el decreto a la lista de mayo.
    decretos_mayo.append(decreto)

# Convierte los decretos de mayo en una tabla.
tabla_mayo = pd.DataFrame(decretos_mayo)

# Muestra todos los registros extraídos.
display(tabla_mayo)

# Comprueba si la cantidad extraída coincide con el total anunciado.
print("¿Coinciden las cantidades?", len(tabla_mayo) == total_mayo)

# Comprueba si existen valores faltantes.
print("Datos faltantes por columna:")
print(tabla_mayo.isna().sum())

,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 054-2024-PCM,29 de mayo de 2024,"Decreto Supremo que prorroga el Estado de Emergencia en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huepetuhe de l ...",https://www.gob.pe/institucion/pcm/normas-legales/5649536-054-2024-pcm
1,Decreto Supremo N.° 052-2024-PCM,22 de mayo de 2024,"Decreto Supremo que declara el Estado de Emergencia en los distritos Pachacamac y Villa María del Triunfo de la provincia de Lima del departamento de Lima, por impacto de daños a consecuencia de expl ...",https://www.gob.pe/institucion/pcm/normas-legales/5649502-052-2024-pcm
2,Decreto Supremo N.° 050-2024-PCM,8 de mayo de 2024,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de la provincia de Quispicanchi del departamento de Cusco, por impacto de daños ante la ocurrencia de movimiento sísmico",https://www.gob.pe/institucion/pcm/normas-legales/5595010-050-2024-pcm


¿Coinciden las cantidades? True
Datos faltantes por columna:
numero    0
fecha     0
titulo    0
enlace    0
dtype: int64


### 6. **Verificación:** impriman una tabla con `mes`, `resultados_totales` y `resultados_extraidos`. Los dos números deben coincidir en todos los meses. Si en alguno no coinciden, revisen su código antes de seguir.

Comparamos, para cada mes, dos cantidades que ya tenemos en Python:

1. **`resultados_totales`:** el número que anuncia gob.pe («N Resultados»), guardado en variables como `total_abril`.
2. **`resultados_extraidos`:** las filas de las tablas que armamos, como `tabla_abril`.

Resultado esperado:

| mes | resultados_totales | resultados_extraidos |
|---|---:|---:|
| Enero | 11 | 11 |
| Febrero | 11 | 11 |
| Marzo | 11 | 11 |
| Abril | 15 | 15 |
| Mayo | 3 | 3 |

In [77]:
# Crea una lista con los resultados anunciados y extraídos de cada mes.
verificacion = [
    {"mes": "Enero", "resultados_totales": total_resultados, "resultados_extraidos": len(tabla_enero)},
    {"mes": "Febrero", "resultados_totales": total_febrero, "resultados_extraidos": len(tabla_febrero)},
    {"mes": "Marzo", "resultados_totales": total_marzo, "resultados_extraidos": len(tabla_marzo)},
    {"mes": "Abril", "resultados_totales": total_abril, "resultados_extraidos": len(tabla_abril)},
    {"mes": "Mayo", "resultados_totales": total_mayo, "resultados_extraidos": len(tabla_mayo)}
]

# Convierte la lista en una tabla de pandas.
tabla_verificacion = pd.DataFrame(verificacion)

# Muestra la tabla completa.
display(tabla_verificacion)

,mes,resultados_totales,resultados_extraidos
0,Enero,11,11
1,Febrero,11,11
2,Marzo,11,11
3,Abril,15,15
4,Mayo,3,3


**¿Coinciden en todos los meses?**

In [78]:
# Compara las dos cantidades de cada mes.
coincidencias = tabla_verificacion["resultados_totales"] == tabla_verificacion["resultados_extraidos"]

# Comprueba si todas las comparaciones son verdaderas.
print("¿Coinciden todos los meses?", coincidencias.all())

¿Coinciden todos los meses? True


### 7. Eliminen las normas repetidas.

**Unimos las tablas de los 5 meses**

In [79]:
# Reúne las cinco tablas mensuales en una sola tabla.
tabla_decretos = pd.concat(
    [tabla_enero, tabla_febrero, tabla_marzo, tabla_abril, tabla_mayo],
    ignore_index=True
)

# Muestra la cantidad de registros reunidos.
print("Registros consolidados:", len(tabla_decretos))

Registros consolidados: 51


**Buscamos normas repetidas** usando como clave el número de la norma (`numero`).

In [80]:
# Identifica todas las filas cuyo número de norma aparece más de una vez.
duplicados = tabla_decretos[
    tabla_decretos.duplicated(subset=["numero"], keep=False)
]

# Muestra cuántos registros tienen un número de norma repetido.
print("Registros con número repetido:", len(duplicados))

# Muestra los registros repetidos, ordenados por número de norma.
display(duplicados.sort_values("numero"))

Registros con número repetido: 0


,numero,fecha,titulo,enlace


### 8. **Ojo:** aunque la búsqueda filtra por PCM, **se cuelan normas de otras instituciones** (por ejemplo, resoluciones del OECE). Quédense solo con las que tienen `/institucion/pcm/` en el enlace e impriman cuántas quitaron.

**Criterio:** revisamos el enlace de cada norma. Si no contiene `/institucion/pcm/`, la excluimos.

In [81]:
# Identifica los enlaces que no contienen la ruta institucional de la PCM.
no_pcm = ~tabla_decretos["enlace"].str.contains("/institucion/pcm/", na=False)

# Selecciona los registros que no pertenecen a la PCM.
tabla_excluidos = tabla_decretos[no_pcm]

# Muestra cuántos registros no corresponden a la PCM.
print("Registros que no pertenecen a la PCM:", len(tabla_excluidos))

# Muestra los registros que tendríamos que excluir.
display(tabla_excluidos)

Registros que no pertenecen a la PCM: 8


,numero,fecha,titulo,enlace
7,Resolución N.° 0214-2024-TCE-S6,18 de enero de 2024,"N° 10304103618); por su responsabilidad al haber contratado con el Estado estando impedida para ello, en el marco de la Orden de Servicio: N° 160-2019-Área de Abastecimiento de fecha 12 de febrero de 2019, emitida po ...",https://www.gob.pe/institucion/oece/normas-legales/5187307-0214-2024-tce-s6
13,Resolución N.° 0634-2024-TCE-S5,23 de febrero de 2024,"Recurso de apelación interpuesto por la empresa SIMED PERU S.A.C., en la ADJUDICACIÓN SIMPLIFICADA Nº 017-2023-HEAV-CS-1 - PRIMERA CONVOCATORIA, para la contratación de suministro de bienes: ""Suministro para el siste ...",https://www.gob.pe/institucion/oece/normas-legales/5260816-0634-2024-tce-s5
14,Resolución N.° 0533-2024-TCE-S4,14 de febrero de 2024,"Recurso de apelación interpuesto por la empresa DIAGNOSTICA PERUANA S.A.C., en el marco de la Licitación Pública Nº 20-2022-ESSALUD-RPR - Primera Convocatoria, convocada por el Seguro Social de Salud, para la ""Adquis ...",https://www.gob.pe/institucion/oece/normas-legales/5197999-0533-2024-tce-s4
17,Resolución N.° 0506-2024-TCE-S5,9 de febrero de 2024,"Recurso de apelación interpuesto por la empresa MEDICALAB SOCIEDAD ANONIMA CERRADA, en el marco de la Adjudicación Simplificada N° 1-2023-CS/HSMSI-1, convocada por el HOSPITAL DE APOYO SANTA MARIA DEL SOCORRO, para l ...",https://www.gob.pe/institucion/oece/normas-legales/5193883-0506-2024-tce-s5
19,Resolución N.° 0437-2024-TCE-S3,5 de febrero de 2024,"Braun Medical Perú S.A. contra el otorgamiento de la buena pro de la Licitación pública N.º 09-2023-HMA-1 – Primera convocatoria, convocado por el Hospital de Emergencias Villa El Salvador",https://www.gob.pe/institucion/oece/normas-legales/5192380-0437-2024-tce-s3
26,Resolución N.° 0991-2024-TCE-S6,21 de marzo de 2024,"Recurso de apelación interpuesto por el postor CORPORACIÓN ALIMENTARIA ALEXCAR S.A.C., en el marco de la Subasta Inversa Electrónica Nº 8-2023-HEJCU-1 (Primera Convocatoria) - Ítem N°s 4, 7, 8 y 9, convocada por el H ...",https://www.gob.pe/institucion/oece/normas-legales/5421041-0991-2024-tce-s6
27,Resolución N.° 0972-2024-TCE-S2,20 de marzo de 2024,"Recurso de apelación interpuesto por la empresa SERVICIOS SANTA GABRIELA S.A.C., en el marco de la Adjudicación Simplificada D.U. 032-2023 N° 065-2023-ENSA - Primera Convocatoria, para la “Adquisición, equipamiento, ...",https://www.gob.pe/institucion/oece/normas-legales/5420398-0972-2024-tce-s2
44,Resolución N.° 1071-2024-TCE-S4,1 de abril de 2024,"Procedimiento administrativo sancionador generado contra el Consorcio Patrón San Marcos, integrado por las empresas CONSTRUCTORA BELCAR E.I.R.L. y J&M ANTARA SERVICIOS GENERALES S.R.L., por su presunta responsabilida ...",https://www.gob.pe/institucion/oece/normas-legales/5464861-1071-2024-tce-s4


**Separamos las normas de la PCM de las excluidas**

In [82]:
# Crea una tabla nueva con los registros que pertenecen a la PCM.
tabla_pcm = tabla_decretos[~no_pcm].copy()

# Crea una tabla independiente con los registros excluidos.
tabla_excluidos = tabla_decretos[no_pcm].copy()

# Muestra la cantidad de registros de cada tabla.
print("Tabla original:", len(tabla_decretos))
print("Tabla PCM:", len(tabla_pcm))
print("Tabla de excluidos:", len(tabla_excluidos))

# Comprueba que no se haya perdido ningún registro.
print("¿Coincide el total?", len(tabla_pcm) + len(tabla_excluidos) == len(tabla_decretos))

Tabla original: 51
Tabla PCM: 43
Tabla de excluidos: 8
¿Coincide el total? True


**Normas de la PCM y normas excluidas**

In [83]:
# Muestra la tabla que contiene únicamente las normas de la PCM.
display(tabla_pcm)

# Muestra las normas excluidas para poder reportarlas.
display(tabla_excluidos)

,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 010-2024-PCM,30 de enero de 2024,"Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrant ...",https://www.gob.pe/institucion/pcm/normas-legales/5398152-010-2024-pcm
1,Decreto Supremo N.° 009-2024-PCM,29 de enero de 2024,"Prorrogar por el termino de sesenta (60) días calendario, a partir del 1 de febrero de 2024, el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la p ...",https://www.gob.pe/institucion/pcm/normas-legales/5398113-009-2024-pcm
2,Decreto Supremo N.° 008-2024-PCM,29 de enero de 2024,"Declarar por el termino de treinta (30) días calendario, a partir del 5 de febrero de 2024 el Estado de Emergencia en el Corredor Vial Sur Apurímac-Cusco-Arequipa, incluyendo los quinientos (500) met ...",https://www.gob.pe/institucion/pcm/normas-legales/5398076-008-2024-pcm
3,Decreto Supremo N.° 007-2024-PCM,29 de enero de 2024,"Decreto Supremo que prorroga el Estado de Emergencia en el distrito de Echarate de la provincia de La Convención del departamento de Cusco, por impacto de daños a consecuencia de intensas pre ...",https://www.gob.pe/institucion/pcm/normas-legales/5225977-007-2024-pcm
4,Decreto Supremo N.° 006-2024-PCM,29 de enero de 2024,"Decreto Supremo que prorroga el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Arequipa, Ayacucho, Cajamarca, Ica, La Libertad, Lambayeque, L ...",https://www.gob.pe/institucion/pcm/normas-legales/5225965-006-2024-pcm
5,Decreto Supremo N.° 005-2024-PCM,29 de enero de 2024,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Arequipa, Ayacucho, Cusco, Huánuco, Huancavelica, Ica, Moquegua y Pasco, por impa ...",https://www.gob.pe/institucion/pcm/normas-legales/5225954-005-2024-pcm
6,Decreto Supremo N.° 004-2024-PCM,26 de enero de 2024,"Decreto Supremo que declara el Estado de Emergencia en el distrito de Chalhuanca de la provincia de Aymaraes del departamento de Apurímac, por impacto de daños a consecuencia de intensas prec ...",https://www.gob.pe/institucion/pcm/normas-legales/5225947-004-2024-pcm
8,Decreto Supremo N.° 003-2024-PCM,10 de enero de 2024,"Decreto Supremo que declara el Estado de Emergencia en la provincia de Zarumilla del departamento de Tumbes; en las provincias de Sullana, Ayabaca y Huancabamba del departamento de Piura; en la provi ...",https://www.gob.pe/institucion/pcm/normas-legales/5225938-003-2024-pcm
9,Decreto Supremo N.° 002-2024-PCM,9 de enero de 2024,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Pataz del departamento de La Libertad,https://www.gob.pe/institucion/pcm/normas-legales/5225914-002-2024-pcm
10,Decreto Supremo N.° 001-2024-PCM,4 de enero de 2024,Decreto Supremo que prorroga el Estado de Emergencia declarado en la Red Vial Nacional y en las Redes Viales Departamentales o Regionales,https://www.gob.pe/institucion/pcm/normas-legales/5225902-001-2024-pcm


,numero,fecha,titulo,enlace
7,Resolución N.° 0214-2024-TCE-S6,18 de enero de 2024,"N° 10304103618); por su responsabilidad al haber contratado con el Estado estando impedida para ello, en el marco de la Orden de Servicio: N° 160-2019-Área de Abastecimiento de fecha 12 de febrero de 2019, emitida po ...",https://www.gob.pe/institucion/oece/normas-legales/5187307-0214-2024-tce-s6
13,Resolución N.° 0634-2024-TCE-S5,23 de febrero de 2024,"Recurso de apelación interpuesto por la empresa SIMED PERU S.A.C., en la ADJUDICACIÓN SIMPLIFICADA Nº 017-2023-HEAV-CS-1 - PRIMERA CONVOCATORIA, para la contratación de suministro de bienes: ""Suministro para el siste ...",https://www.gob.pe/institucion/oece/normas-legales/5260816-0634-2024-tce-s5
14,Resolución N.° 0533-2024-TCE-S4,14 de febrero de 2024,"Recurso de apelación interpuesto por la empresa DIAGNOSTICA PERUANA S.A.C., en el marco de la Licitación Pública Nº 20-2022-ESSALUD-RPR - Primera Convocatoria, convocada por el Seguro Social de Salud, para la ""Adquis ...",https://www.gob.pe/institucion/oece/normas-legales/5197999-0533-2024-tce-s4
17,Resolución N.° 0506-2024-TCE-S5,9 de febrero de 2024,"Recurso de apelación interpuesto por la empresa MEDICALAB SOCIEDAD ANONIMA CERRADA, en el marco de la Adjudicación Simplificada N° 1-2023-CS/HSMSI-1, convocada por el HOSPITAL DE APOYO SANTA MARIA DEL SOCORRO, para l ...",https://www.gob.pe/institucion/oece/normas-legales/5193883-0506-2024-tce-s5
19,Resolución N.° 0437-2024-TCE-S3,5 de febrero de 2024,"Braun Medical Perú S.A. contra el otorgamiento de la buena pro de la Licitación pública N.º 09-2023-HMA-1 – Primera convocatoria, convocado por el Hospital de Emergencias Villa El Salvador",https://www.gob.pe/institucion/oece/normas-legales/5192380-0437-2024-tce-s3
26,Resolución N.° 0991-2024-TCE-S6,21 de marzo de 2024,"Recurso de apelación interpuesto por el postor CORPORACIÓN ALIMENTARIA ALEXCAR S.A.C., en el marco de la Subasta Inversa Electrónica Nº 8-2023-HEJCU-1 (Primera Convocatoria) - Ítem N°s 4, 7, 8 y 9, convocada por el H ...",https://www.gob.pe/institucion/oece/normas-legales/5421041-0991-2024-tce-s6
27,Resolución N.° 0972-2024-TCE-S2,20 de marzo de 2024,"Recurso de apelación interpuesto por la empresa SERVICIOS SANTA GABRIELA S.A.C., en el marco de la Adjudicación Simplificada D.U. 032-2023 N° 065-2023-ENSA - Primera Convocatoria, para la “Adquisición, equipamiento, ...",https://www.gob.pe/institucion/oece/normas-legales/5420398-0972-2024-tce-s2
44,Resolución N.° 1071-2024-TCE-S4,1 de abril de 2024,"Procedimiento administrativo sancionador generado contra el Consorcio Patrón San Marcos, integrado por las empresas CONSTRUCTORA BELCAR E.I.R.L. y J&M ANTARA SERVICIOS GENERALES S.R.L., por su presunta responsabilida ...",https://www.gob.pe/institucion/oece/normas-legales/5464861-1071-2024-tce-s4


### 9. Los títulos de la lista aparecen **cortados** ("..."), y justo lo que se corta suele ser el motivo (*"por peligro inminente..."*). La página de cada norma **no necesita Selenium**: léanla con `requests` y `BeautifulSoup` *(Revisar Sesión 4)* y saquen el **título completo**. Hagan una pausa de 1 segundo entre páginas.

> Pistas: (1) gob.pe **rechaza** los pedidos que no se identifican como navegador; usen `requests.get(enlace, headers={"User-Agent": "Mozilla/5.0"})`. (2) El título completo está en `sopa.find("div", class_="description")`. Al final trae un texto extra (*"DS N° ... PDF ... Descargar"*) que pueden dejar o limpiar.

**Cómo lo hacemos:** la página de cada norma es HTML estático, así que basta con `requests`, enviando un `User-Agent` de navegador porque gob.pe rechaza los pedidos que no lo parecen. El título completo está en `<div class="description">`. Ese bloque también trae la ficha del PDF (*"DS N° ... PDF 1.5 MB Descargar"*), que eliminamos para que no llegue al CSV.

In [84]:
import time
from bs4 import BeautifulSoup


def titulo_completo(enlace):
    """Devuelve el título completo de una norma desde su página, o None si no se pudo leer."""
    time.sleep(1)  # indicación: pausa de 1 segundo entre páginas
    r = requests.get(enlace, headers=headers, timeout=30)  # headers: User-Agent de navegador (paso 1)
    div = BeautifulSoup(r.text, "html.parser").find("div", class_="description")
    if div is None:  # gob.pe rechazó el pedido: la página no trae el bloque
        return None
    ficha_pdf = div.find(class_="institution-document__files")  # "DS N° ... PDF ... Descargar"
    if ficha_pdf:
        ficha_pdf.decompose()  # se quita para que no llegue al CSV
    return " ".join(div.get_text(" ", strip=True).split())  # un solo espacio entre palabras


tabla = tabla_pcm.rename(columns={"titulo": "titulo_corto"})  # el título cortado queda para comparar
tabla["titulo"] = tabla["enlace"].apply(titulo_completo)  # 43 páginas: tarda cerca de 1 minuto

In [85]:
# Verificación: ningún título vacío, cortado ni con la ficha del PDF
t = tabla["titulo"]
cortado = tabla["titulo_corto"].str.endswith("...")
print("Normas:", len(t),
      "| sin título:", t.isna().sum(),
      "| terminan en '...':", t.str.endswith("...", na=False).sum(),
      "| con 'Descargar':", t.str.contains("Descargar", na=False).sum(),
      "| cortados que se completaron:", (t[cortado].str.len() > tabla.loc[cortado, "titulo_corto"].str.len()).sum(),
      "de", cortado.sum())

# Ejemplo del mismo título antes y después
print("\nCortado: ", tabla.loc[cortado, "titulo_corto"].iloc[0])
print("Completo:", tabla.loc[cortado, "titulo"].iloc[0])

Normas: 43 | sin título: 0 | terminan en '...': 0 | con 'Descargar': 0 | cortados que se completaron: 28 de 28

Cortado:  Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrant ...
Completo: Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrante del presente Decreto Supremo.


### 10. Creen estas columnas usando el título completo en minúsculas (`.lower()`) y el operador `in`:

- `es_lluvia`: `True` si el título contiene `"lluvia"` o `"precipitaciones"`.
- `tipo`: `"prorroga"` si contiene `"prórroga"` o `"prorroga"`; si no, `"declara"` si contiene `"declara"`; si no, `"otro"`.
- `motivo`: `"peligro inminente"` o `"impacto de daños"` si el título contiene esa frase; si no, `"otro"`.

> Revisen las filas que quedaron como `"otro"` y, en una celda de texto, expliquen si están bien clasificadas.

In [86]:
# Indicación: título completo en minúsculas (.lower()) y operador in
def es_lluvia_titulo(t):
    t = t.lower()
    return "lluvia" in t or "precipitaciones" in t


def tipo_titulo(t):
    t = t.lower()
    if "prórroga" in t or "prorroga" in t:  # va primero: las prórrogas también dicen "declarado"
        return "prorroga"
    return "declara" if "declara" in t else "otro"


def motivo_titulo(t):
    t = t.lower()
    for frase in ["peligro inminente", "impacto de daños"]:
        if frase in t:
            return frase
    return "otro"


assert tabla["titulo"].notna().all(), "Faltan títulos: repetir el paso 9"
tabla["es_lluvia"] = tabla["titulo"].apply(es_lluvia_titulo)
tabla["tipo"] = tabla["titulo"].apply(tipo_titulo)
tabla["motivo"] = tabla["titulo"].apply(motivo_titulo)

for col in ["es_lluvia", "tipo", "motivo"]:
    print(tabla[col].value_counts(), "\n")

es_lluvia
False    23
True     20
Name: count, dtype: int64 

tipo
prorroga    26
declara     17
Name: count, dtype: int64 

motivo
otro                 21
impacto de daños     20
peligro inminente     2
Name: count, dtype: int64 



In [87]:
pd.set_option("display.max_colwidth", None)  # muestra los títulos sin cortar

# Indicación: revisar las filas "otro". La tabla cruzada muestra si alguna es de lluvias.
display(pd.crosstab(tabla["motivo"], tabla["es_lluvia"]))
display(tabla.loc[(tabla["tipo"] == "otro") | (tabla["motivo"] == "otro"), ["numero", "tipo", "motivo", "titulo"]])

# Control de tipo: prórrogas cuyo título no empieza así (¿una declaratoria que solo menciona una prórroga?)
inicio = tabla["titulo"].str[:80].str.lower()
display(tabla.loc[(tabla["tipo"] == "prorroga") & ~inicio.str.contains("prorrog|prórrog"), ["numero", "titulo"]])

es_lluvia,False,True
motivo,,
impacto de daños,2,18
otro,21,0
peligro inminente,0,2


,numero,tipo,motivo,titulo
0,Decreto Supremo N.° 010-2024-PCM,prorroga,otro,"Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrante del presente Decreto Supremo."
1,Decreto Supremo N.° 009-2024-PCM,prorroga,otro,"Prorrogar por el termino de sesenta (60) días calendario, a partir del 1 de febrero de 2024, el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huepetuhe de la provincia de Manu del departamento de Madre de Dios. La Policía Nacional del Perú mantiene el control del orden interno, con el apoyo de las Fuerzas Armadas."
2,Decreto Supremo N.° 008-2024-PCM,declara,otro,"Declarar por el termino de treinta (30) días calendario, a partir del 5 de febrero de 2024 el Estado de Emergencia en el Corredor Vial Sur Apurímac-Cusco-Arequipa, incluyendo los quinientos (500) metros adyacentes a cada lado de la via. La Policía Nacional del Perú mantiene el control del orden interno, con el apoyo de las Fuerzas Armadas."
8,Decreto Supremo N.° 003-2024-PCM,declara,otro,"Decreto Supremo que declara el Estado de Emergencia en la provincia de Zarumilla del departamento de Tumbes; en las provincias de Sullana, Ayabaca y Huancabamba del departamento de Piura; en la provincia de San Ignacio del departamento de Cajamarca; en el distrito de El Cenepa de la provincia de Condorcanqui del departamento de Amazonas y en el distrito de Torres Causana de la provincia de Maynas del departamento de Loreto"
9,Decreto Supremo N.° 002-2024-PCM,prorroga,otro,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Pataz del departamento de La Libertad
10,Decreto Supremo N.° 001-2024-PCM,prorroga,otro,Decreto Supremo que prorroga el Estado de Emergencia declarado en la Red Vial Nacional y en las Redes Viales Departamentales o Regionales
15,Decreto Supremo N.° 019-2024-PCM,declara,otro,Decreto Supremo que declara el Estado de Emergencia en la provincia de Trujillo del departamento de La Libertad
16,Decreto Supremo N.° 018-2024-PCM,declara,otro,Decreto Supremo que declara el Estado de Emergencia en la provincia de Pataz del departamento de La Libertad
20,Decreto Supremo N.° 013-2024-PCM,prorroga,otro,Decreto Supremo que prorroga el Estado de Emergencia en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto
22,Decreto Supremo N.° 034-2024-PCM,prorroga,otro,"Decreto Supremo que prorroga el Estado de Emergencia en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huepetuhe de la provincia de Manu del departamento de Madre de Dios"


,numero,titulo


**¿Están bien clasificadas las filas `"otro"`?**

- **`tipo`:** ninguna norma quedó como `"otro"` (26 prórrogas y 17 declaratorias). La única prórroga cuyo título no empieza con esa palabra es el DS 037-2024-PCM (*"Prórroga del Estado de Emergencia…"*), así que está bien clasificada.
- **`motivo`:** 21 normas quedaron como `"otro"` y la tabla cruzada muestra que **ninguna es de lluvias**. Son emergencias de otro tipo: Pataz, Trujillo, Zarumilla, Putumayo, Tambopata y Manu, el Corredor Vial Sur o la Red Vial Nacional. Varias dicen que *"la Policía Nacional del Perú mantiene el control del orden interno"*. Sus títulos no mencionan "peligro inminente" ni "impacto de daños", así que están **bien clasificadas**.
- **Caso dudoso:** el DS 010-2024-PCM prorroga el estado de emergencia *"en los distritos que se indican en el Anexo"* sin decir el motivo. Podría ser de lluvias, pero el título no permite saberlo; es una limitación de clasificar solo con el título.
- El DS 050-2024-PCM (sismo) y el DS 052-2024-PCM (explosión) dicen "impacto de daños", pero no son de lluvias: `es_lluvia` los descarta en el paso 11.

### 11. Quédense solo con las normas de lluvias. Impriman cuántas son y muestren 5 filas.

In [88]:
# Indicación: solo normas de lluvias; imprimir cuántas son y mostrar 5 filas
lluvias = tabla[tabla["es_lluvia"]].reset_index(drop=True)
print("Normas de lluvias:", len(lluvias), "| descartadas:", len(tabla) - len(lluvias))
display(lluvias[["numero", "fecha", "tipo", "motivo", "titulo"]].head(5))

Normas de lluvias: 20 | descartadas: 23


,numero,fecha,tipo,motivo,titulo
0,Decreto Supremo N.° 007-2024-PCM,29 de enero de 2024,prorroga,impacto de daños,"Decreto Supremo que prorroga el Estado de Emergencia en el distrito de Echarate de la provincia de La Convención del departamento de Cusco, por impacto de daños a consecuencia de intensas precipitaciones pluviales"
1,Decreto Supremo N.° 006-2024-PCM,29 de enero de 2024,prorroga,peligro inminente,"Decreto Supremo que prorroga el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Arequipa, Ayacucho, Cajamarca, Ica, La Libertad, Lambayeque, Lima, Pasco, Piura, San Martín y Tumbes, por peligro inminente ante intensas precipitaciones pluviales (período 2023- 2024) y posible Fenómeno El Niño"
2,Decreto Supremo N.° 005-2024-PCM,29 de enero de 2024,declara,impacto de daños,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Arequipa, Ayacucho, Cusco, Huánuco, Huancavelica, Ica, Moquegua y Pasco, por impacto de daños a consecuencia de intensas precipitaciones pluviales"
3,Decreto Supremo N.° 004-2024-PCM,26 de enero de 2024,declara,impacto de daños,"Decreto Supremo que declara el Estado de Emergencia en el distrito de Chalhuanca de la provincia de Aymaraes del departamento de Apurímac, por impacto de daños a consecuencia de intensas precipitaciones pluviales"
4,Decreto Supremo N.° 021-2024-PCM,28 de febrero de 2024,declara,impacto de daños,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Arequipa, Ica, Madre de Dios, Moquegua y Ucayali, por impacto de daños a consecuencia de intensas precipitaciones pluviales"


### 12. Identifiquen **qué departamentos menciona cada decreto**. Un decreto puede mencionar varios. Usen esta lista:

```python
departamentos = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
                 "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
                 "Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
                 "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
                 "San Martín", "Tacna", "Tumbes", "Ucayali"]
```
> Ojo: si buscan en el título en minúsculas, `"ica"` aparece dentro de **huancavel*ica***, y un decreto que solo menciona Huancavelica contaría también para Ica. Busquen en el título original (con mayúsculas) y por palabra completa: `re.search(r"\b" + departamento + r"\b", titulo)`. Además, "Ica" también es una **provincia**: si un decreto dice *"provincia de Ica del departamento de Ica"*, debe contar **una sola vez**. En una celda de texto, expliquen con un ejemplo de su temporada cómo evitaron contar mal.

In [89]:
import unicodedata

departamentos = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
                 "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
                 "Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
                 "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
                 "San Martín", "Tacna", "Tumbes", "Ucayali"]

# Un nombre que sigue a "provincia de" o "distrito de" no es el departamento:
# "provincia de Ucayali del departamento de Loreto" cuenta para Loreto, no para Ucayali.
NO_ES_DEPARTAMENTO = r"(?<!provincia de )(?<!provincias de )(?<!distrito de )(?<!distritos de )"


def sin_tildes(texto):
    """'Áncash' -> 'Ancash'. Algunos títulos escriben los nombres sin tilde."""
    return "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")


def departamentos_en(titulo):
    """Departamentos que menciona el título. Cada uno cuenta una sola vez aunque se repita."""
    # Indicación: título original (con mayúsculas, sin .lower()) y palabra completa (\b)
    t = sin_tildes(titulo)
    return [d for d in departamentos if re.search(NO_ES_DEPARTAMENTO + r"\b" + sin_tildes(d) + r"\b", t)]


lluvias["departamentos"] = lluvias["titulo"].apply(departamentos_en)
lluvias["n_departamentos"] = lluvias["departamentos"].apply(len)
display(lluvias[["numero", "tipo", "departamentos"]])

,numero,tipo,departamentos
0,Decreto Supremo N.° 007-2024-PCM,prorroga,[Cusco]
1,Decreto Supremo N.° 006-2024-PCM,prorroga,"[Amazonas, Áncash, Arequipa, Ayacucho, Cajamarca, Ica, La Libertad, Lambayeque, Lima, Pasco, Piura, San Martín, Tumbes]"
2,Decreto Supremo N.° 005-2024-PCM,declara,"[Arequipa, Ayacucho, Cusco, Huancavelica, Huánuco, Ica, Moquegua, Pasco]"
3,Decreto Supremo N.° 004-2024-PCM,declara,[Apurímac]
4,Decreto Supremo N.° 021-2024-PCM,declara,"[Arequipa, Ica, Madre de Dios, Moquegua, Ucayali]"
5,Decreto Supremo N.° 020-2024-PCM,declara,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Loreto, Moquegua, Pasco, Puno]"
6,Decreto Supremo N.° 016-2024-PCM,declara,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cusco, Huancavelica, Huánuco, Junín, Lima, Madre de Dios, Pasco, Puno, San Martín, Tacna]"
7,Decreto Supremo N.° 0012-2024-PCM,declara,"[Áncash, Apurímac, Cusco, Huancavelica, Lima, Loreto, San Martín]"
8,Decreto Supremo N.° 033-2024-PCM,prorroga,"[Arequipa, Ayacucho, Cusco, Huancavelica, Huánuco, Ica, Moquegua, Pasco]"
9,Decreto Supremo N.° 032-2024-PCM,prorroga,"[Áncash, Apurímac, Cusco, Huancavelica, Lima, Loreto, San Martín]"


In [90]:
def diferencia(otro, correcto):
    """Lo que otro método agrega (+) o pierde (-) frente al nuestro."""
    return ", ".join([f"+{d}" for d in otro if d not in correcto] + [f"-{d}" for d in correcto if d not in otro])


# Verificación: ¿en qué decretos se equivocarían dos métodos más simples?
# - minusculas: el error que advierte el enunciado ("ica" dentro de "huancavelica")
# - solo_b: palabra completa, pero sin quitar tildes ni descartar provincias
minusculas = lluvias["titulo"].apply(lambda t: [d for d in departamentos if d.lower() in t.lower()])
solo_b = lluvias["titulo"].apply(lambda t: [d for d in departamentos if re.search(r"\b" + d + r"\b", t)])
comparacion = pd.DataFrame({
    "numero": lluvias["numero"],
    "minusculas": [diferencia(o, c) for o, c in zip(minusculas, lluvias["departamentos"])],
    "solo_palabra_completa": [diferencia(o, c) for o, c in zip(solo_b, lluvias["departamentos"])],
})
display(comparacion[(comparacion["minusculas"] != "") | (comparacion["solo_palabra_completa"] != "")])

# Indicación: un departamento nombrado dos veces (como "Ica" provincia y departamento) cuenta una sola vez
dos_veces = lluvias["titulo"].apply(lambda t: [d for d in departamentos if len(re.findall(r"\b" + d + r"\b", t)) > 1])
display(lluvias.loc[dos_veces.apply(len) > 0, ["numero", "departamentos"]].assign(nombrado_dos_veces=dos_veces))
print("Decretos sin departamento:", (lluvias["n_departamentos"] == 0).sum())

,numero,minusculas,solo_palabra_completa
6,Decreto Supremo N.° 016-2024-PCM,+Ica,
7,Decreto Supremo N.° 0012-2024-PCM,"+Ica, -Áncash, -San Martín","-Áncash, -San Martín"
9,Decreto Supremo N.° 032-2024-PCM,+Ica,
12,Decreto Supremo N.° 027-2024-PCM,+Ica,
13,Decreto Supremo N.° 026-2024-PCM,+Ica,
14,Decreto Supremo N.° 049-2024-PCM,+Ucayali,+Ucayali
17,Decreto Supremo N.° 044-2024-PCM,+Ica,
18,Decreto Supremo N.° 039-2024-PCM,+Ica,


,numero,departamentos,nombrado_dos_veces
16,Decreto Supremo N.° 045-2024-PCM,[Arequipa],[Arequipa]


Decretos sin departamento: 0


**¿Cómo evitamos contar mal? Ejemplos de nuestra temporada (2024)**

- **"Ica" dentro de "Huancavelica":** el DS 016-2024-PCM nombra Huancavelica, pero no Ica. Buscando en minúsculas, "ica" aparece dentro de "huancavel**ica**" y se sumaría Ica; con `\b` no pasa.
- **Nombrado dos veces:** el DS 045-2024-PCM dice *"provincia de Arequipa del departamento de Arequipa"*. Como preguntamos una vez por cada departamento, Arequipa cuenta **una sola vez**. Ningún decreto de 2024 repite "Ica", pero la regla es la misma.
- **Provincia con nombre de otro departamento:** el DS 049-2024-PCM menciona la *"provincia de Ucayali del departamento de Loreto"*. Con solo `\b` se contaba Ucayali; al descartar los nombres que siguen a "provincia de", cuenta solo para Ayacucho y Loreto.
- **Sin tilde:** el DS 0012-2024-PCM escribe "Ancash" y "San Martin". Sin quitar las tildes, esos dos departamentos se perdían.

### 13. Guarden dos archivos:

- `datos/decretos_lluvias.csv` con las columnas `numero`, `fecha`, `titulo`, `tipo`, `motivo` y `enlace`.
- `datos/decretos_por_departamento.csv` con las columnas `departamento`, `declaratorias` y `prorrogas`.

> Pista: guarden con `to_csv(..., index=False, encoding="utf-8-sig")` para que las tildes se vean bien si abren el archivo en Excel.

In [91]:
import os

os.makedirs("datos", exist_ok=True)

# Columnas pedidas; index=False y utf-8-sig según la pista del enunciado
lluvias[["numero", "fecha", "titulo", "tipo", "motivo", "enlace"]].to_csv(
    "datos/decretos_lluvias.csv", index=False, encoding="utf-8-sig")

# explode: una fila por (decreto, departamento). reset_index evita el índice repetido que hace fallar a crosstab.
menciones = lluvias.explode("departamentos").reset_index(drop=True)
por_dep = (pd.crosstab(menciones["departamentos"], menciones["tipo"])
           .reindex(index=departamentos, columns=["declara", "prorroga"], fill_value=0)  # 25 filas, 0 si no hay
           .rename(columns={"declara": "declaratorias", "prorroga": "prorrogas"})
           .rename_axis(index="departamento", columns=None)
           .reset_index())

# Controles: 25 departamentos y todas las menciones decreto-departamento contadas
assert len(por_dep) == 25
assert por_dep[["declaratorias", "prorrogas"]].to_numpy().sum() == lluvias["n_departamentos"].sum()

por_dep.to_csv("datos/decretos_por_departamento.csv", index=False, encoding="utf-8-sig")
display(por_dep)

,departamento,declaratorias,prorrogas
0,Amazonas,4,2
1,Áncash,5,5
2,Apurímac,7,4
3,Arequipa,7,6
4,Ayacucho,7,5
5,Cajamarca,3,3
6,Callao,0,0
7,Cusco,7,5
8,Huancavelica,7,4
9,Huánuco,6,3


In [92]:
# Verificación: releer los archivos guardados y revisar filas y columnas
for archivo in ["decretos_lluvias", "decretos_por_departamento"]:
    df = pd.read_csv(f"datos/{archivo}.csv", encoding="utf-8-sig")
    print(f"{archivo}.csv: {len(df)} filas | columnas: {list(df.columns)}")

decretos_lluvias.csv: 20 filas | columnas: ['numero', 'fecha', 'titulo', 'tipo', 'motivo', 'enlace']
decretos_por_departamento.csv: 25 filas | columnas: ['departamento', 'declaratorias', 'prorrogas']
